# Kaggriculture RL v2 — PPO trained on the *real* environment

A from-scratch rebuild of `Kaggriculture_RL_LION.ipynb`. Every block below is explained in the markdown above it and can be run and inspected on its own.

## What was wrong with v1, and what this notebook does differently

| v1 problem | v2 fix |
|---|---|
| Trained on a homemade CuPy toy simulator with fabricated prices and rules | Trains directly against the **real** `kaggriculture` environment |
| Shaping rewards (plant +2, water +0.5, harvest +10, sell "dopamine") taught reward-hacking | Reward = **Δbank per step only** — it telescopes exactly to final profit |
| Shaping annealed to 0 → policy collapsed to "buy everything, sell nothing" by epoch 130 | Nothing to anneal; profit is the objective from step 0 |
| Action masks applied at rollout but **dropped in the PPO update** | Masks stored per step and re-applied to logits inside the update |
| Monte-Carlo returns, no value bootstrap, critic loss in the millions | **GAE(λ=0.95)**, bootstrap at rollout end, reward scaling, grad clipping |
| `strict=False` checkpoint loads, hyperparameters changed mid-resume | One canonical config cell; strict `state_dict` loads |
| Inference read obs keys that don't exist (`obs['bank']`, `obs['farm_grid']`) → constant zero input | Encoder parses the real observation format (`farms`, `private`, `market`, `town`) |
| Submission emitted `["w_0 WATER"]` strings → silent no-ops | Decoder emits the real `{"farmer", "hands", "market"}` dict |
| Deployment action map misaligned with training indices | One shared action table used by training, eval, and submission |

## Architecture at a glance

* **Factorized policy**: a *farmer head* (movement + farm ops) and a *market head* (buy seeds / fertilizer / land), both action-masked every turn.
* **Scripted selling**: every turn, all sellable shed produce is auto-sold. Harvest → cash next turn, so the Δbank reward is dense and tightly coupled to good farming.
* **Out of scope for v2.0** (extension points are marked): animals, hired hands, learned sell-timing.


## 0. Setup & config

One canonical config dict — no mid-run hyperparameter edits like v1's resume cell.

**`SMOKE = True` (default)** runs everything tiny (4 envs, 4-day episodes, 2 epochs) so you can test each cell quickly and watch the whole pipeline work end-to-end in a couple of minutes. **Set `SMOKE = False` for the real training run** (16 envs, full 720-turn seasons, 300 epochs — a few hours on CPU, faster on GPU).

In [ ]:
# On Colab, uncomment these two lines first:
# !pip install -U kaggle-environments
# from google.colab import drive; drive.mount('/content/drive')

import os
import time
import json
import math
import inspect
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from kaggle_environments import make

# ---- the one switch for step-by-step testing -------------------------------
SMOKE = True   # <- set to False for the full training run
# -----------------------------------------------------------------------------

CONFIG = dict(
    # environment
    n_envs         = 4    if SMOKE else 16,   # parallel real-env instances
    episode_steps  = 96   if SMOKE else 720,  # 720 = full 30-day season
    starting_money = 3000.0,
    # ppo
    epochs         = 3    if SMOKE else 300,
    gamma          = 0.999,
    lam            = 0.95,
    clip_eps       = 0.2,
    ppo_epochs     = 2    if SMOKE else 4,
    mini_batch     = 1024 if SMOKE else 4096,
    lr             = 3e-4,
    ent_coef       = 0.005,
    vf_coef        = 0.5,
    max_grad_norm  = 0.5,
    reward_scale   = 0.01,      # Δbank is in dollars -> O(0.01..1) rewards
    # behavior-cloning warm start
    bc_episodes    = 1    if SMOKE else 12,   # expert episodes per env
    bc_epochs      = 1    if SMOKE else 5,
    bc_lr          = 1e-3,
    use_bc_init    = True,
    # opponents are sampled per episode from this pool
    opponent_pool  = ["starter", "random"],
    log_every      = 1 if SMOKE else 5,
    ckpt_every     = 1 if SMOKE else 25,
)

# Where checkpoints / replays / the submission bundle go.
# On Colab, point this at your Drive: '/content/drive/MyDrive/kaggriculture_rl'
WORKDIR = os.getcwd()
CKPT_DIR = os.path.join(WORKDIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
print("device:", device, "| SMOKE:", SMOKE, "| workdir:", WORKDIR)

## 1. Game constants & the observation encoder

This is the fix for v1's most basic failure: v1's inference code read `obs['bank']`, `obs['farm_grid']`, `obs['workers']` — **keys that do not exist in the real observation**, so the network received the same all-zero tensor on every turn and could only ever emit one constant action.

`encode_obs` parses the **real** format documented in the README:

* `obs["farms"][player]["tiles"][y][x]` → 15 spatial channels (10×10): empty / locked / weed, one channel per crop, plus plant age, watered-today, consecutive-unwatered, harvestable yield, fertilized, coop, pasture.
* money, day/hour clock, farmer position, seeds, shed contents, farmer inventory, market prices, market inventory deviation, unlocked quadrants, opponent money (public), town shops → a 48-dim global vector.

Everything is roughly normalized to [0, 1] so the network starts with well-scaled inputs.

In [ ]:
CROPS    = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
ANIMALS  = ["GOOSE", "COW", "SHEEP"]
SEED_COST = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
LAND_PRICES = [1000, 2000, 4000]                    # NE, SW, SE unlock order
BOARD = 10
SHED_ADJACENT = {(4, 4), (5, 4), (4, 5), (5, 5)}    # (x, y) center tiles, boardSize=10
SELLABLE = [p for p in PRODUCTS if p != "FERTILIZER"]  # auto-sell keeps fertilizer stock

N_SPATIAL = 15
N_GLOBAL  = 48

def encode_obs(obs):
    """Real kaggriculture obs dict -> (spatial [15,10,10] f32, globals [48] f32)."""
    player = obs["player"]
    farm = obs["farms"][player]
    opp  = obs["farms"][1 - player]
    day  = obs["day"]

    s = np.zeros((N_SPATIAL, BOARD, BOARD), dtype=np.float32)
    for y in range(BOARD):
        for x in range(BOARD):
            t = farm["tiles"][y][x]
            if t is None:
                s[0, y, x] = 1.0                              # empty unlocked tile
            elif t == "LOCKED":
                s[1, y, x] = 1.0                              # locked quadrant tile
            elif isinstance(t, dict):
                kind = t.get("kind")
                if kind == "WEED":
                    s[2, y, x] = 1.0
                elif kind == "PLANT":
                    s[3 + CROPS.index(t["crop"]), y, x] = 1.0  # crop one-hot, ch 3-7
                    s[8, y, x]  = min((day - t["planted_day"]) / 16.0, 1.0)
                    s[9, y, x]  = 1.0 if t["watered_today"] else 0.0
                    s[10, y, x] = min(t["consecutive_unwatered"] / 2.0, 1.0)
                    s[11, y, x] = min(t["yield_units"] / 6.0, 1.0)
                    s[12, y, x] = 1.0 if t.get("fertilized_until_day", -1) >= day else 0.0
                elif kind == "COOP":
                    s[13, y, x] = 1.0
                elif kind == "PASTURE":
                    s[14, y, x] = 1.0

    fx, fy = farm["farmer"]
    g = np.zeros(N_GLOBAL, dtype=np.float32)
    g[0] = farm["money"] / 5000.0
    g[1] = day / 30.0
    g[2] = obs["hour"] / 24.0
    g[3] = fx / (BOARD - 1)
    g[4] = fy / (BOARD - 1)
    for i, c in enumerate(CROPS):                                   # 5-9: seed counts
        g[5 + i] = min(obs["private"]["seeds"].get(c, 0) / 10.0, 2.0)
    for i, p in enumerate(PRODUCTS):                                # 10-18: shed products
        g[10 + i] = min(obs["private"]["shed"].get(p, 0) / 20.0, 2.0)
    for i, a in enumerate(ANIMALS):                                 # 19-21: shed animals
        g[19 + i] = obs["private"]["shed"].get(a, 0) / 4.0
    finv = obs["private"]["inventories"][0] if obs["private"]["inventories"] else {}
    g[22] = min(sum(finv.values()) / 10.0, 2.0)                     # farmer inventory
    g[23] = finv.get("FERTILIZER", 0) / 5.0
    g[24] = finv.get("WHEAT", 0) / 10.0
    for i, p in enumerate(PRODUCTS):                                # 25-33: market prices
        g[25 + i] = obs["market"]["prices"][p] / 250.0
    for i, p in enumerate(PRODUCTS):                                # 34-42: inventory deviation
        g[34 + i] = (obs["market"]["inventory"][p] - 10000.0) / 2000.0
    g[43] = len(farm["unlocked_quadrants"]) / 4.0
    g[44] = opp["money"] / 5000.0                                   # public info
    g[45] = len(obs["town"]["unlocked_shops"]) / 8.0
    g[46] = 1.0 if (fx, fy) in SHED_ADJACENT else 0.0
    g[47] = min(farm.get("hires_today", 0) / 5.0, 1.0)
    return s, g

### ✅ Test the encoder on a real observation

Run a fresh env, encode the initial observation, and eyeball the output. You should see the farmer starting at (4, 4) with $3,000, only the NW quadrant unlocked (channel 1 = LOCKED everywhere except the top-left 5×5), and base market prices.

In [ ]:
_env = make("kaggriculture", configuration={"episodeSteps": 96, "seed": 1})
_env.reset(num_agents=2)
_obs = _env.state[0].observation

_s, _g = encode_obs(_obs)
print("spatial:", _s.shape, " globals:", _g.shape)
print("farmer pos (global[3:5]):", _g[3] * 9, _g[4] * 9, " money:", _g[0] * 5000)
print("locked tiles:", int(_s[1].sum()), "(expect 75 = 3 locked quadrants)")
print("empty tiles:",  int(_s[0].sum()), "(expect 25)")
print("wheat price feature:", _g[25], "->", _g[25] * 250, "(expect 25)")
print("obs keys:", list(_obs.keys()))

## 2. Action space, masks, and decoding

v1's submission mapped action indices to the **wrong commands** (its training index 15 meant *buy melon seed*, its deployment map said *buy goose*) and emitted strings the real interpreter silently discarded. Here there is **one action table** shared by training, evaluation, and the submission.

Two factorized heads, both masked every step:

* **Farmer head (15 actions)** — 4 moves, PASS, WATER, HARVEST, DIG, 5 × PLANT_\<crop\>, FERTILIZE, PICKUP_FERTILIZER.
  *(Animal ops — BUILD_COOP/PASTURE, PLACE, FEED, CARE, COLLECT_FERTILIZER — are the v2.1 extension point.)*
* **Market head (8 actions)** — NOOP, 5 × BUY_SEED, BUY_PRODUCT FERTILIZER, BUY_LAND.

**Selling is scripted, not learned**: `build_action_dict` always appends SELL orders for everything sellable in the shed (up to the 10-orders/turn cap). In v1 the agent had to *learn* to sell and never did; here harvest → cash on the next turn, which is what makes the Δbank reward dense.

`action_masks` computes exactly which actions are legal right now (e.g. WATER only on an unwatered plant, PLANT only on empty land with seeds in stock, BUY_LAND only when affordable) — invalid actions are impossible to sample, so the policy never wastes probability mass on no-ops.

In [ ]:
FARMER_ACTIONS = (
    ["NORTH", "SOUTH", "EAST", "WEST", "PASS", "WATER", "HARVEST", "DIG"]   # 0-7
    + [f"PLANT_{c}" for c in CROPS]                                        # 8-12
    + ["FERTILIZE", "PICKUP_FERTILIZER"]                                   # 13-14
)
MARKET_ACTIONS = (["NOOP"]                                                 # 0
    + [f"BUY_SEED_{c}" for c in CROPS]                                     # 1-5
    + ["BUY_FERTILIZER", "BUY_LAND"])                                      # 6-7
N_FARMER = len(FARMER_ACTIONS)
N_MARKET = len(MARKET_ACTIONS)
print(N_FARMER, "farmer actions:", FARMER_ACTIONS)
print(N_MARKET, "market actions:", MARKET_ACTIONS)

def action_masks(obs):
    """-> (farmer_mask [15] bool, market_mask [8] bool) for a real obs."""
    player = obs["player"]
    farm = obs["farms"][player]
    fx, fy = farm["farmer"]
    tile = farm["tiles"][fy][fx]
    money = farm["money"]
    seeds = obs["private"]["seeds"]
    shed  = obs["private"]["shed"]
    finv  = obs["private"]["inventories"][0] if obs["private"]["inventories"] else {}

    fm = np.zeros(N_FARMER, dtype=bool)
    fm[0] = fy > 0                            # NORTH  (moving off the board is a wasted no-op -> mask it)
    fm[1] = fy < BOARD - 1                    # SOUTH
    fm[2] = fx < BOARD - 1                    # EAST
    fm[3] = fx > 0                            # WEST
    fm[4] = True                              # PASS

    is_plant = isinstance(tile, dict) and tile.get("kind") == "PLANT"
    is_weed  = isinstance(tile, dict) and tile.get("kind") == "WEED"
    fm[5] = is_plant and not tile["watered_today"]                # WATER
    fm[6] = is_plant and tile["yield_units"] > 0                  # HARVEST
    fm[7] = is_plant or is_weed                                   # DIG
    for i, c in enumerate(CROPS):                                 # PLANT_<crop>
        fm[8 + i] = (tile is None) and seeds.get(c, 0) > 0
    fm[13] = (is_plant and finv.get("FERTILIZER", 0) > 0          # FERTILIZE
              and tile.get("fertilized_until_day", -1) < obs["day"])
    fm[14] = ((fx, fy) in SHED_ADJACENT                           # PICKUP_FERTILIZER
              and shed.get("FERTILIZER", 0) > 0)

    mm = np.zeros(N_MARKET, dtype=bool)
    mm[0] = True                                                  # NOOP
    for i, c in enumerate(CROPS):                                 # BUY_SEED_<crop>
        mm[1 + i] = money >= SEED_COST[c] and seeds.get(c, 0) < 30
    mm[6] = money >= obs["market"]["prices"]["FERTILIZER"] and sum(shed.values()) < 100
    n_quads = len(farm["unlocked_quadrants"])
    mm[7] = n_quads < 4 and money >= LAND_PRICES[n_quads - 1]     # BUY_LAND
    return fm, mm

def build_action_dict(f_idx, m_idx, obs):
    """(farmer idx, market idx) + obs -> real env action dict, with scripted auto-sell."""
    shed = obs["private"]["shed"]

    name = FARMER_ACTIONS[f_idx]
    if name.startswith("PLANT_"):
        farmer = ["PLANT", name.split("_", 1)[1]]
    elif name == "PICKUP_FERTILIZER":
        farmer = ["PICKUP", "FERTILIZER", 1]
    else:
        farmer = [name]

    market = []
    mname = MARKET_ACTIONS[m_idx]
    if mname.startswith("BUY_SEED_"):
        market.append(["BUY_SEED", mname.split("_", 2)[2], 1])
    elif mname == "BUY_FERTILIZER":
        market.append(["BUY_PRODUCT", "FERTILIZER", 1])
    elif mname == "BUY_LAND":
        market.append(["BUY_LAND"])

    # scripted auto-sell: dump every sellable product in the shed (<= 10 orders/turn total)
    for p in SELLABLE:
        if len(market) >= 10:
            break
        n = shed.get(p, 0)
        if n > 0:
            market.append(["SELL", p, int(n)])

    return {"farmer": farmer, "hands": [], "market": market}

# Built-in agents vary: some take (obs, config), others just (obs). Detect once.
def call_agent(fn, obs, config):
    try:
        n = len(inspect.signature(fn).parameters)
    except (TypeError, ValueError):
        n = 2
    return fn(obs, config) if n >= 2 else fn(obs)

### ✅ Test masks and the action decoder

On the opening observation the farmer stands at (4, 4) — a shed-adjacent tile — with no seeds: expect moves + PASS + no plants, and the market mask to allow buying any affordable seed. Then we actually **play** a buy + plant + water sequence into the real env to prove the decoder produces legal actions (v1 never checked this — its actions were silently dropped as invalid).

In [ ]:
_fm, _mm = action_masks(_obs)
print("farmer mask:", [FARMER_ACTIONS[i] for i in np.where(_fm)[0]])
print("market mask:", [MARKET_ACTIONS[i] for i in np.where(_mm)[0]])

# play a real sequence: BUY_SEED WHEAT -> PLANT -> WATER
_a = build_action_dict(4, 1, _obs)     # PASS + BUY_SEED_WHEAT (+ auto-sell: shed is empty)
print("turn 1 action:", _a)
_env.step([_a, {"farmer": ["PASS"], "hands": [], "market": []}])
_obs = _env.state[0].observation
print("seeds after buy:", _obs["private"]["seeds"], "money:", _obs["farms"][0]["money"])

_a = build_action_dict(8, 0, _obs)     # PLANT_WHEAT + NOOP
print("turn 2 action:", _a)
_env.step([_a, {"farmer": ["PASS"], "hands": [], "market": []}])
_obs = _env.state[0].observation
_fx, _fy = _obs["farms"][0]["farmer"]
print("tile under farmer:", _obs["farms"][0]["tiles"][_fy][_fx])
_fm, _mm = action_masks(_obs)
print("farmer mask now:", [FARMER_ACTIONS[i] for i in np.where(_fm)[0]], "(WATER should appear)")
assert all(s.status == "ACTIVE" for s in _env.state), "an action was invalid!"

## 3. Model

A small conv trunk over the 10×10 board plus an MLP over the globals, fused in a shared torso with three heads: farmer logits, market logits, scalar value.

The policy is factorized: `log π(a) = log π_farmer(a_f) + log π_market(a_m)`. `masked_categorical` sets invalid-action logits to −1e9 **before** the softmax — the same helper is used during rollouts *and* inside the PPO update, which fixes v1's bug of computing probability ratios against an unmasked distribution.

In [ ]:
class FarmPolicy(nn.Module):
    def __init__(self, n_spatial=N_SPATIAL, n_global=N_GLOBAL,
                 n_farmer=N_FARMER, n_market=N_MARKET):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(n_spatial, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.mlp = nn.Sequential(nn.Linear(n_global, 64), nn.ReLU())
        torso_in = 64 * BOARD * BOARD + 64
        self.torso = nn.Sequential(nn.Linear(torso_in, 512), nn.ReLU())
        self.farmer_head = nn.Linear(512, n_farmer)
        self.market_head = nn.Linear(512, n_market)
        self.value_head  = nn.Linear(512, 1)

    def forward(self, spatial, glob):
        z = torch.cat([self.conv(spatial), self.mlp(glob)], dim=1)
        z = self.torso(z)
        return self.farmer_head(z), self.market_head(z), self.value_head(z).squeeze(-1)

def masked_categorical(logits, mask):
    return torch.distributions.Categorical(logits=logits.masked_fill(~mask, -1e9))

@torch.no_grad()
def act(model, spatial, glob, fmask, mmask, greedy=False):
    """Sample (or argmax) a factorized action under the masks. Returns cpu-friendly tensors."""
    f_logits, m_logits, v = model(spatial, glob)
    fd = masked_categorical(f_logits, fmask)
    md = masked_categorical(m_logits, mmask)
    fa = fd.logits.argmax(-1) if greedy else fd.sample()
    ma = md.logits.argmax(-1) if greedy else md.sample()
    logp = fd.log_prob(fa) + md.log_prob(ma)
    ent  = fd.entropy() + md.entropy()
    return fa, ma, logp, v, ent

_m = FarmPolicy().to(device)
print(f"parameters: {sum(p.numel() for p in _m.parameters()):,}")

## 4. Vectorized runner over *real* environments

`VecKag` steps N real `kaggriculture` instances in lockstep (~1 ms per env-step, so 16 envs ≈ a 35 s epoch rollout). Design points:

* **Reward = Δbank × `reward_scale`, and nothing else.** No shaping to hack, nothing to anneal — the sum of per-step rewards over an episode *is* the final profit.
* Each env is **recreated on reset with a fresh `seed`** so weed spawns and town-shop unlocks vary across episodes (the interpreter consumes the seed at episode start, so a fresh `make()` per episode is the reliable way to reseed).
* The **opponent plays as player 1** (`starter` / `random` from the pool). Its sells move the shared market, so the policy learns against real price dynamics.
* When an episode ends, the env auto-resets and the final bank balance is recorded into `final_moneys` — our true training metric.

In [ ]:
class VecKag:
    def __init__(self, n_envs, cfg, opponent_pool):
        self.n = n_envs
        self.cfg = cfg
        self.opponent_pool = opponent_pool
        self.episode_count = [0] * n_envs
        self.opponents = [opponent_pool[i % len(opponent_pool)] for i in range(n_envs)]
        self.envs = [None] * n_envs
        self.prev_money = np.zeros(n_envs, dtype=np.float64)
        self.final_moneys = []          # rolling record of completed episodes
        self._opp_cache = {}

    def _opp_fn(self, env, name):
        if name not in self._opp_cache:
            self._opp_cache[name] = env.agents[name] if isinstance(name, str) else name
        return self._opp_cache[name]

    def _make_env(self, i):
        conf = {"episodeSteps": self.cfg["episode_steps"],
                "seed": 1000 * self.episode_count[i] + 7 * i + 1}
        env = make("kaggriculture", configuration=conf)
        env.reset(num_agents=2)
        return env

    def reset(self):
        for i in range(self.n):
            self.envs[i] = self._make_env(i)
            self.episode_count[i] += 1
        self.prev_money[:] = self.cfg["starting_money"]
        return self._observe_all()

    def _observe_all(self):
        S, G, FM, MM = [], [], [], []
        for i, env in enumerate(self.envs):
            obs = env.state[0].observation
            s, g = encode_obs(obs)
            fm, mm = action_masks(obs)
            S.append(s); G.append(g); FM.append(fm); MM.append(mm)
        return (torch.tensor(np.stack(S)), torch.tensor(np.stack(G)),
                torch.tensor(np.stack(FM)), torch.tensor(np.stack(MM)))

    def step(self, f_actions, m_actions):
        rewards = np.zeros(self.n, dtype=np.float32)
        dones = np.zeros(self.n, dtype=bool)
        for i, env in enumerate(self.envs):
            obs0 = env.state[0].observation
            a0 = build_action_dict(int(f_actions[i]), int(m_actions[i]), obs0)
            try:
                a1 = call_agent(self._opp_fn(env, self.opponents[i]),
                                env.state[1].observation, env.configuration)
                if not isinstance(a1, dict):
                    a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            except Exception:
                a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            env.step([a0, a1])
            money = env.state[0].observation["farms"][0]["money"]
            rewards[i] = (money - self.prev_money[i]) * self.cfg["reward_scale"]
            self.prev_money[i] = money
            if env.state[0].status != "ACTIVE":
                dones[i] = True
                self.final_moneys.append(money)
                self.envs[i] = self._make_env(i)          # auto-reset, fresh seed
                self.episode_count[i] += 1
                self.prev_money[i] = self.cfg["starting_money"]
        return self._observe_all(), torch.tensor(rewards), torch.tensor(dones, dtype=torch.float32)

### ✅ Test the runner

Five steps of random *masked* actions. Rewards should be small negatives (seed purchases) or zero, and no exception / INVALID status should occur.

In [ ]:
_vec = VecKag(CONFIG["n_envs"], CONFIG, CONFIG["opponent_pool"])
_S, _G, _FM, _MM = _vec.reset()
print("batch:", _S.shape, _G.shape, _FM.shape, _MM.shape)
for _t in range(5):
    _fa = torch.stack([torch.tensor(np.random.choice(np.where(m.numpy())[0])) for m in _FM])
    _ma = torch.stack([torch.tensor(np.random.choice(np.where(m.numpy())[0])) for m in _MM])
    (_S, _G, _FM, _MM), _r, _d = _vec.step(_fa, _ma)
    print(f"t={_t}  reward={_r.numpy().round(3).tolist()}  done={_d.tolist()}")
del _vec

## 5. Behavior-cloning warm start (optional but recommended)

Δbank is dense enough for PPO to learn from scratch, but the opening of the game is a chicken-and-egg problem: buying seeds is an immediate *negative* reward whose payoff lands 3+ days (~75 turns) later. A warm start skips that cold start.

The built-in **`starter` bot** (a single-tile carrot loop that finishes ≈ $3,488) is called directly on real observations; its action dicts are translated into our factorized indices, and we train both heads with masked cross-entropy. Because `starter` only ever plays one carrot tile, BC alone caps out at ≈ $3,488 — PPO's job is to go beyond it (more tiles, better crops, land expansion).

Set `use_bc_init = False` in the config to skip this and run pure PPO.

In [ ]:
def expert_labels(act_d):
    """starter's action dict -> (farmer_idx, market_idx) in our action tables."""
    fop = act_d.get("farmer", ["PASS"])
    fi = 4                                                    # PASS
    if fop[0] in ("NORTH", "SOUTH", "EAST", "WEST", "WATER", "HARVEST", "DIG", "FERTILIZE"):
        fi = FARMER_ACTIONS.index(fop[0])
    elif fop[0] == "PLANT" and len(fop) > 1 and fop[1] in CROPS:
        fi = 8 + CROPS.index(fop[1])
    elif fop[0] == "PICKUP" and len(fop) > 1 and fop[1] == "FERTILIZER":
        fi = FARMER_ACTIONS.index("PICKUP_FERTILIZER")
    mi = 0                                                    # NOOP (sells are scripted)
    for order in act_d.get("market", []):
        if order[0] == "BUY_SEED" and order[1] in CROPS:
            mi = 1 + CROPS.index(order[1]); break
        if order[0] == "BUY_PRODUCT" and order[1] == "FERTILIZER":
            mi = 6; break
        if order[0] == "BUY_LAND":
            mi = 7; break
    return fi, mi

def collect_bc_data(vec, n_episodes_per_env):
    """Roll the envs forward by *playing* the expert, labeling every step."""
    starter = None
    S_buf, G_buf, FM_buf, MM_buf, FA_buf, MA_buf = [], [], [], [], [], []
    S, G, FM, MM = vec.reset()
    target_steps = n_episodes_per_env * vec.cfg["episode_steps"]
    for _ in range(target_steps):
        fa = torch.zeros(vec.n, dtype=torch.long)
        ma = torch.zeros(vec.n, dtype=torch.long)
        for i, env in enumerate(vec.envs):
            if starter is None:
                starter = env.agents["starter"]
            fi, mi = expert_labels(call_agent(starter, env.state[0].observation, env.configuration))
            fa[i], ma[i] = fi, mi
        S_buf.append(S); G_buf.append(G); FM_buf.append(FM); MM_buf.append(MM)
        FA_buf.append(fa); MA_buf.append(ma)
        (S, G, FM, MM), _, _ = vec.step(fa, ma)
    return (torch.stack(S_buf), torch.stack(G_buf), torch.stack(FM_buf),
            torch.stack(MM_buf), torch.stack(FA_buf), torch.stack(MA_buf))

def behavior_clone(model, vec, cfg):
    print("collecting BC data from the 'starter' expert ...")
    t0 = time.time()
    S, G, FM, MM, FA, MA = collect_bc_data(vec, cfg["bc_episodes"])
    n = S.shape[0] * S.shape[1]
    print(f"  {n} labeled steps in {time.time()-t0:.0f}s; expert final money: "
          f"{np.mean(vec.final_moneys[-n_ep_safeguard(vec):] or [0]):.0f}")
    opt = torch.optim.Adam(model.parameters(), lr=cfg["bc_lr"])
    bS  = S.view(n, N_SPATIAL, BOARD, BOARD).to(device)
    bG  = G.view(n, N_GLOBAL).to(device)
    bFM = FM.view(n, N_FARMER).to(device)
    bMM = MM.view(n, N_MARKET).to(device)
    bFA = FA.view(-1).to(device)
    bMA = MA.view(-1).to(device)
    model.train()
    for ep in range(cfg["bc_epochs"]):
        perm = torch.randperm(n, device=device)
        tot_l = tot_a = cnt = 0
        for st in range(0, n, cfg["mini_batch"]):
            idx = perm[st:st + cfg["mini_batch"]]
            f_logits, m_logits, _ = model(bS[idx], bG[idx])
            fl = f_logits.masked_fill(~bFM[idx], -1e9)
            ml = m_logits.masked_fill(~bMM[idx], -1e9)
            loss = F.cross_entropy(fl, bFA[idx]) + F.cross_entropy(ml, bMA[idx])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            opt.step()
            tot_l += loss.item(); cnt += 1
            tot_a += ((fl.argmax(-1) == bFA[idx]).float().mean()
                      + (ml.argmax(-1) == bMA[idx]).float().mean()).item() / 2
        print(f"  BC epoch {ep}: loss {tot_l/cnt:.4f}  acc {tot_a/cnt:.3f}")
    path = os.path.join(CKPT_DIR, "bc_init.pt")
    torch.save(model.state_dict(), path)
    print("  saved", path)

def n_ep_safeguard(vec):
    return max(1, len(vec.final_moneys))

### ▶️ Create the model + envs, run the BC warm start

After BC, the greedy policy should roughly reproduce the starter bot: buy a carrot seed, plant it, water it, harvest it. The printed expert final money (≈ $3,488 in full seasons) is the baseline PPO must beat.

In [ ]:
model = FarmPolicy().to(device)
vec = VecKag(CONFIG["n_envs"], CONFIG, CONFIG["opponent_pool"])

if CONFIG["use_bc_init"]:
    behavior_clone(model, vec, CONFIG)
else:
    print("BC skipped (use_bc_init=False)")

## 6. PPO training loop

Point-by-point repair of v1's update:

| | v1 | v2 |
|---|---|---|
| Advantage | full 720-step Monte-Carlo return, no bootstrap | **GAE(λ=0.95)** with value bootstrap at rollout end |
| Masks | applied at rollout, **absent in the update** | stored per step, **re-applied to logits in the update** |
| Value targets | O(100,000) → critic loss in the millions | Δbank×0.01 → targets O(1–30) |
| Gradients | no clipping, Lion @ wd 1e-2 | `clip_grad_norm_(0.5)`, Adam 3e-4 |
| Metric | "Avg Step Reward" (dominated by shaping) | **mean final money** per completed episode |

Watch the `final$` column: below $3,000 = worse than doing nothing; $3,488 = starter-bot level; above that = genuinely learning.

In [ ]:
def train_ppo(model, vec, cfg):
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    T, N = cfg["episode_steps"], vec.n
    hist = {"final_money": [], "entropy": [], "kl": [], "vloss": []}

    S, G, FM, MM = vec.reset()
    for epoch in range(cfg["epochs"]):
        t0 = time.time()
        sp  = torch.zeros(T, N, N_SPATIAL, BOARD, BOARD)
        gl  = torch.zeros(T, N, N_GLOBAL)
        fmk = torch.zeros(T, N, N_FARMER, dtype=torch.bool)
        mmk = torch.zeros(T, N, N_MARKET, dtype=torch.bool)
        fac = torch.zeros(T, N, dtype=torch.long)
        mac = torch.zeros(T, N, dtype=torch.long)
        lgp = torch.zeros(T, N)
        val = torch.zeros(T, N)
        rew = torch.zeros(T, N)
        don = torch.zeros(T, N)

        model.eval()
        for t in range(T):
            sp[t], gl[t], fmk[t], mmk[t] = S, G, FM, MM
            fa, ma, logp, v, _ = act(model, S.to(device), G.to(device),
                                     FM.to(device), MM.to(device))
            fac[t], mac[t], lgp[t], val[t] = fa.cpu(), ma.cpu(), logp.cpu(), v.cpu()
            (S, G, FM, MM), r, d = vec.step(fa.cpu(), ma.cpu())
            rew[t], don[t] = r, d

        # ---- GAE with bootstrap -------------------------------------------
        with torch.no_grad():
            _, _, v_last = model(S.to(device), G.to(device))
        v_last = v_last.cpu()
        adv = torch.zeros(T, N)
        lastgae = torch.zeros(N)
        for t in reversed(range(T)):
            nonterm = 1.0 - don[t]
            nextv = v_last if t == T - 1 else val[t + 1]
            delta = rew[t] + cfg["gamma"] * nextv * nonterm - val[t]
            lastgae = delta + cfg["gamma"] * cfg["lam"] * nonterm * lastgae
            adv[t] = lastgae
        ret = adv + val

        # ---- PPO update ----------------------------------------------------
        B = T * N
        bsp, bgl = sp.view(B, N_SPATIAL, BOARD, BOARD), gl.view(B, N_GLOBAL)
        bfmk, bmmk = fmk.view(B, N_FARMER), mmk.view(B, N_MARKET)
        bfac, bmac, blgp = fac.view(-1), mac.view(-1), lgp.view(-1)
        badv, bret = adv.view(-1), ret.view(-1)
        badv = (badv - badv.mean()) / (badv.std() + 1e-8)

        model.train()
        ep_ent = ep_kl = ep_vl = nb = 0
        for _ in range(cfg["ppo_epochs"]):
            perm = torch.randperm(B)
            for st in range(0, B, cfg["mini_batch"]):
                idx = perm[st:st + cfg["mini_batch"]]
                f_logits, m_logits, v = model(bsp[idx].to(device), bgl[idx].to(device))
                fd = masked_categorical(f_logits, bfmk[idx].to(device))   # masks RE-APPLIED
                md = masked_categorical(m_logits, bmmk[idx].to(device))
                new_logp = fd.log_prob(bfac[idx].to(device)) + md.log_prob(bmac[idx].to(device))
                ratio = torch.exp(new_logp - blgp[idx].to(device))
                a_i = badv[idx].to(device)
                surr = torch.min(ratio * a_i,
                                 torch.clamp(ratio, 1 - cfg["clip_eps"], 1 + cfg["clip_eps"]) * a_i)
                actor_loss = -surr.mean()
                v_loss = F.mse_loss(v, bret[idx].to(device))
                ent = (fd.entropy() + md.entropy()).mean()
                loss = actor_loss + cfg["vf_coef"] * v_loss - cfg["ent_coef"] * ent
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                opt.step()
                ep_ent += ent.item(); ep_vl += v_loss.item()
                ep_kl += (blgp[idx].to(device) - new_logp).mean().item(); nb += 1

        hist["entropy"].append(ep_ent / nb)
        hist["kl"].append(ep_kl / nb)
        hist["vloss"].append(ep_vl / nb)
        recent = vec.final_moneys[-2 * N:] or [cfg["starting_money"]]
        hist["final_money"].append(float(np.mean(recent)))

        if epoch % cfg["log_every"] == 0:
            print(f"ep {epoch:03d} | final$ {np.mean(recent):8.0f} | ent {ep_ent/nb:5.2f} "
                  f"| kl {ep_kl/nb:+.4f} | vloss {ep_vl/nb:7.3f} | {time.time()-t0:.0f}s")
        if (epoch + 1) % cfg["ckpt_every"] == 0:
            p = os.path.join(CKPT_DIR, f"ppo_ep{epoch+1}.pt")
            torch.save(model.state_dict(), p)
            print("  -> checkpoint", p)
    return hist

### ▶️ Run PPO

In SMOKE mode this is 3 tiny epochs — you're checking that it *runs* and that the loss/KL numbers are finite, not that it plays well. For the real run (SMOKE=False, ~2–4 h on CPU) expect `final$` to climb past $3,000 in the first few dozen epochs and past $3,488 after that.

In [ ]:
history = train_ppo(model, vec, CONFIG)
final_path = os.path.join(CKPT_DIR, "ppo_final.pt")
torch.save(model.state_dict(), final_path)
print("saved", final_path)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(history["final_money"])
    ax[0].axhline(3000, ls="--", c="gray", label="do-nothing")
    ax[0].axhline(3488, ls="--", c="green", label="starter bot")
    ax[0].set_title("Mean final money"); ax[0].legend()
    ax[1].plot(history["entropy"]); ax[1].set_title("Policy entropy")
    ax[2].plot(history["kl"]); ax[2].set_title("Approx KL / update")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; history dict:", {k: [round(x, 3) for x in v] for k, v in history.items()})

## 7. Greedy evaluation vs the built-in bots

The real scoreboard: full 720-turn seasons, greedy (argmax) policy, against `starter` and `random`. Also writes an HTML replay so you can *watch* the agent farm — open the file in a browser.

Note these run full seasons regardless of `SMOKE` (evaluation must use the real game length), so in SMOKE mode it plays just 1–2 games and the untrained-ish policy will likely score poorly — that's expected.

In [ ]:
@torch.no_grad()
def evaluate(model, opponent, n_games=10, render_first=False):
    model.eval()
    results = []
    for game in range(n_games):
        env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 5000 + game})
        env.reset(num_agents=2)
        opp = env.agents[opponent]
        while env.state[0].status == "ACTIVE":
            obs0 = env.state[0].observation
            s, g = encode_obs(obs0)
            fm, mm = action_masks(obs0)
            fa, ma, _, _, _ = act(model,
                                  torch.tensor(s).unsqueeze(0).to(device),
                                  torch.tensor(g).unsqueeze(0).to(device),
                                  torch.tensor(fm).unsqueeze(0).to(device),
                                  torch.tensor(mm).unsqueeze(0).to(device),
                                  greedy=True)
            a0 = build_action_dict(int(fa[0]), int(ma[0]), obs0)
            try:
                a1 = call_agent(opp, env.state[1].observation, env.configuration)
                if not isinstance(a1, dict):
                    a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            except Exception:
                a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            env.step([a0, a1])
        results.append(env.state[0].observation["farms"][0]["money"])
        if game == 0 and render_first:
            out = os.path.join(WORKDIR, f"replay_vs_{opponent}.html")
            with open(out, "w") as f:
                f.write(env.render(mode="html"))
            print("  replay written to", out)
    print(f"vs {opponent:8s}: mean ${np.mean(results):.0f} "
          f"(min ${min(results):.0f}, max ${max(results):.0f}) over {n_games} games")
    return results

if SMOKE:
    evaluate(model, "starter", n_games=1, render_first=True)
else:
    evaluate(model, "starter", n_games=10, render_first=True)
    evaluate(model, "random",  n_games=10)

## 8. Package the Kaggle submission

`submission.py` embeds **verbatim copies** of `encode_obs`, `action_masks`, `build_action_dict`, the action tables, and the model class — pulled via `inspect.getsource`, so the deployed agent is guaranteed to encode/decode exactly like training (v1's train/serve skew is impossible by construction).

The agent returns the **real** action-dict format `{"farmer", "hands", "market"}`.

Upload with:
```bash
kaggle competitions submit kaggriculture -f submission.tar.gz -m "PPO v2"
```

In [ ]:
def write_submission(out_py):
    template = """
import os
import numpy as np
import torch
import torch.nn as nn

CROPS    = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
ANIMALS  = ["GOOSE", "COW", "SHEEP"]
SEED_COST = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
LAND_PRICES = [1000, 2000, 4000]
BOARD = 10
SHED_ADJACENT = {(4, 4), (5, 4), (4, 5), (5, 5)}
SELLABLE = [p for p in PRODUCTS if p != "FERTILIZER"]
N_SPATIAL = 15
N_GLOBAL  = 48
FARMER_ACTIONS = (
    ["NORTH", "SOUTH", "EAST", "WEST", "PASS", "WATER", "HARVEST", "DIG"]
    + [f"PLANT_{c}" for c in CROPS]
    + ["FERTILIZE", "PICKUP_FERTILIZER"]
)
MARKET_ACTIONS = ["NOOP"] + [f"BUY_SEED_{c}" for c in CROPS] + ["BUY_FERTILIZER", "BUY_LAND"]
N_FARMER = len(FARMER_ACTIONS)
N_MARKET = len(MARKET_ACTIONS)

__ENCODER_SRC__

__MASKS_SRC__

__DECODE_SRC__

class FarmPolicy(nn.Module):
    def __init__(self, n_spatial=N_SPATIAL, n_global=N_GLOBAL,
                 n_farmer=N_FARMER, n_market=N_MARKET):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(n_spatial, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.mlp = nn.Sequential(nn.Linear(n_global, 64), nn.ReLU())
        torso_in = 64 * BOARD * BOARD + 64
        self.torso = nn.Sequential(nn.Linear(torso_in, 512), nn.ReLU())
        self.farmer_head = nn.Linear(512, n_farmer)
        self.market_head = nn.Linear(512, n_market)
        self.value_head  = nn.Linear(512, 1)

    def forward(self, spatial, glob):
        z = torch.cat([self.conv(spatial), self.mlp(glob)], dim=1)
        z = self.torso(z)
        return self.farmer_head(z), self.market_head(z), self.value_head(z).squeeze(-1)

_model = None

def _load():
    global _model
    if _model is None:
        _model = FarmPolicy()
        for cand in [os.path.join(os.path.dirname(os.path.abspath(__file__)), "model.pth"),
                     "model.pth",
                     "/kaggle_simulations/agent/model.pth"]:
            if os.path.exists(cand):
                _model.load_state_dict(torch.load(cand, map_location="cpu"))
                break
        else:
            print("WARNING: model.pth not found; running an untrained net")
        _model.eval()
    return _model

def agent(obs, config):
    model = _load()
    s, g = encode_obs(obs)
    fm, mm = action_masks(obs)
    with torch.no_grad():
        f_logits, m_logits, _ = model(torch.tensor(s).unsqueeze(0),
                                      torch.tensor(g).unsqueeze(0))
        fa = f_logits.masked_fill(~torch.tensor(fm).unsqueeze(0), -1e9).argmax(-1).item()
        ma = m_logits.masked_fill(~torch.tensor(mm).unsqueeze(0), -1e9).argmax(-1).item()
    return build_action_dict(fa, ma, obs)
"""
    code_text = (template
                 .replace("__ENCODER_SRC__", inspect.getsource(encode_obs))
                 .replace("__MASKS_SRC__", inspect.getsource(action_masks))
                 .replace("__DECODE_SRC__", inspect.getsource(build_action_dict)))
    with open(out_py, "w") as f:
        f.write(code_text)
    print("wrote", out_py)

submission_py = os.path.join(WORKDIR, "submission.py")
write_submission(submission_py)

import shutil, tarfile
model_pth = os.path.join(WORKDIR, "model.pth")
shutil.copyfile(final_path, model_pth)
tar_path = os.path.join(WORKDIR, "submission.tar.gz")
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(submission_py, arcname="submission.py")
    tar.add(model_pth, arcname="model.pth")
print("packaged", tar_path)

### ✅ Submission smoke test — the check v1 never had

Runs the packaged `submission.py` through a **full 720-turn match** in the real env and asserts both players reach `DONE`. If our action format were malformed the interpreter would mark the agent `INVALID` or `ERROR` — this cell fails loudly instead of silently scoring 0 on Kaggle.

In [ ]:
def smoke_test_submission():
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 999}, debug=True)
    env.run([os.path.join(WORKDIR, "submission.py"), "starter"])
    st = env.state
    for i in range(2):
        print(f"player {i}: status={st[i].status} reward={st[i].reward}")
    assert all(st[i].status == "DONE" for i in range(2)), "agent crashed or sent invalid actions!"
    print("submission agent is legal and completes a full match.")

smoke_test_submission()

## 9. Where to go next (v2.1+)

1. **Animals** (highest value): add `BUILD_COOP`/`BUILD_PASTURE`, `PLACE_GOOSE/...`, `FEED`, `CARE`, `COLLECT_FERTILIZER`, `PICKUP_WHEAT` to `FARMER_ACTIONS` and `BUY_ANIMAL_*` to `MARKET_ACTIONS`. The obs channels and mask scaffolding already exist — a goose coop feeding on purchased wheat is the strongest economy line in the game.
2. **Hired hands**: a HIRE market action plus a per-hand head (or broadcast the farmer head over each hand's local view). Labor is the main throughput limiter late-game.
3. **Learned sell timing**: replace scripted auto-sell with a per-product sell head. This matters for melon/strawberry/milk/wool, whose prices crash to $1 on gluts — batching and timing sales is where the premium-goods money is.
4. **League self-play**: freeze snapshots of the policy into `opponent_pool`. The market is shared, so the game is adversarial even without direct contact.
5. **Throughput**: move `VecKag` to multiprocessing past ~32 envs; the Python interpreter (~1 ms/step) is the bottleneck, not the GPU.
